# ECS647U/773P — Bayesian Decision and Risk Analysis
## Coursework: Bayesian and Decision Modelling with pyAgrum
### Domain: California Housing Market Analysis
### Dataset: California Housing Prices (1990 US Census)



## Setup and Installation

In [ ]:
# Install pyAgrum (required for all modelling tasks)
!pip install pyAgrum -q


In [ ]:
# Q1-Q7: Import all required libraries
import pyAgrum as gum
import pyAgrum.lib.notebook as gnb
import pyAgrum.causal as csl
import pyAgrum.causal.notebook as cslnb

import pandas as pd
import numpy as np
from sklearn.datasets import fetch_california_housing
import warnings
warnings.filterwarnings('ignore')

print(f"pyAgrum version: {gum.__version__}")
gnb.configuration()


## Question 1: Problem Definition and Variable Design
This section loads and prepares the California Housing dataset for BN modelling.


In [ ]:
# Q1: Load the California Housing dataset from scikit-learn
data = fetch_california_housing(as_frame=True)
df = data.frame.copy()
print(f"Dataset shape: {df.shape}")
print(f"\nFeature descriptions:")
for i, name in enumerate(data.feature_names):
    print(f"  {name}: {data.DESCR.split('- ')[i+1].split(':')[0] if i < 8 else name}")
print(f"\nTarget: MedHouseVal (median house value)")
df.head()


In [ ]:
# Q1: Select the 6 variables for our BN model
# Rename columns for clarity in the BN
selected = df[['MedInc', 'HouseAge', 'AveRooms', 'AveOccup', 'Population', 'MedHouseVal']].copy()
selected.columns = ['Income', 'HouseAge', 'RoomSize', 'Occupancy', 'Population', 'HouseValue']

print(f"Selected variables: {list(selected.columns)}")
print(f"Records: {len(selected)}")
print(f"Missing values: {selected.isnull().sum().sum()}")
print(f"\nSummary statistics:")
selected.describe()


In [ ]:
# Q1: Discretise all continuous variables into 3 categories using quantile-based binning
# This ensures balanced distributions across states (approx 33% each)

disc = pd.DataFrame()

# Income: Low / Medium / High
disc['Income'] = pd.qcut(selected['Income'], q=3, labels=['Low', 'Medium', 'High'])

# HouseAge: New / Mid / Old
disc['HouseAge'] = pd.qcut(selected['HouseAge'], q=3, labels=['New', 'Mid', 'Old'], duplicates='drop')

# RoomSize: Small / Medium / Large
disc['RoomSize'] = pd.qcut(selected['RoomSize'], q=3, labels=['Small', 'Medium', 'Large'])

# Occupancy: Low / Medium / High
disc['Occupancy'] = pd.qcut(selected['Occupancy'], q=3, labels=['Low', 'Medium', 'High'])

# Population: Sparse / Moderate / Dense
disc['Population'] = pd.qcut(selected['Population'], q=3, labels=['Sparse', 'Moderate', 'Dense'])

# HouseValue: Cheap / Moderate / Expensive
disc['HouseValue'] = pd.qcut(selected['HouseValue'], q=3, labels=['Cheap', 'Moderate', 'Expensive'])

# Print distribution of each variable
print(f"Missing values: {disc.isnull().sum().sum()}")
for col in disc.columns:
    print(f"\n{col}:")
    print(disc[col].value_counts(normalize=True).sort_index())

# Save to CSV for pyAgrum
disc.to_csv('california_housing_disc.csv', index=False)
print(f"\nDiscretised dataset saved to california_housing_disc.csv")


## Question 2: Network Structure and Justification
Learn the BN structure from the discretised dataset using Greedy Hill Climbing with BDeu scoring.


In [ ]:
# Q2: Structure learning using Greedy Hill Climbing with BDeu score
learner = gum.BNLearner('california_housing_disc.csv')

# Use Greedy Hill Climbing algorithm
learner.useGreedyHillClimbing()

# Add mandatory arcs based on domain knowledge:
# Income -> HouseValue (income drives property prices)
# Income -> RoomSize (income determines housing quality/size)
learner.addMandatoryArc('Income', 'HouseValue')
learner.addMandatoryArc('Income', 'RoomSize')

# Print learner configuration
print(learner)

# Learn the BN (both structure and parameters via MLE)
bn = learner.learnBN()

print(f"\nLearned BN: {bn.size()} nodes, {bn.sizeArcs()} arcs")
print(f"\nArcs:")
for arc in bn.arcs():
    tail = bn.variable(arc[0]).name()
    head = bn.variable(arc[1]).name()
    print(f"  {tail} -> {head}")


In [ ]:
# Q2: Display the learned BN structure
gnb.showBN(bn, size="8")


In [ ]:
# Q2: Show the BN with inference (prior distributions)
gnb.showInference(bn, size="10")


In [ ]:
# Q2: Identify root, leaf, and intermediate nodes
root_nodes = [bn.variable(n).name() for n in bn.nodes() if bn.parents(n) == set()]
leaf_nodes = [bn.variable(n).name() for n in bn.nodes() if bn.children(n) == set()]
intermediate_nodes = [bn.variable(n).name() for n in bn.nodes()
                      if bn.parents(n) != set() and bn.children(n) != set()]

print(f"Root nodes (no parents): {root_nodes}")
print(f"Leaf nodes (no children): {leaf_nodes}")
print(f"Intermediate nodes: {intermediate_nodes}")

# Show parent info for each node
for n in bn.nodes():
    name = bn.variable(n).name()
    parents = [bn.variable(p).name() for p in bn.parents(n)]
    children = [bn.variable(c).name() for c in bn.children(n)]
    n_params = bn.cpt(n).domainSize() // bn.variable(n).domainSize() * bn.variable(n).domainSize()
    print(f"\n{name}:")
    print(f"  Parents: {parents if parents else 'None (root)'}")
    print(f"  Children: {children if children else 'None (leaf)'}")
    print(f"  CPT parameters: {bn.cpt(n).domainSize()}")


## Question 3: Parameterisation of CPTs
Present CPTs for: (1) root node, (2) leaf node, (3) intermediate node, (4) multi-parent node.
All CPTs were learned from data using Maximum Likelihood Estimation.


In [ ]:
# Q3: Root node — Income (no parents)
print("=" * 50)
print("ROOT NODE: Income")
print("=" * 50)
print(f"Parents: None")
print(f"Children: {[bn.variable(c).name() for c in bn.children(bn.idFromName('Income'))]}")
print(f"Number of parameters: {bn.cpt('Income').domainSize()}")
print(f"\nCPT of Income (root node - no parents):")
gnb.flow.row(bn.cpt("Income"), captions=["CPT of Income"])


In [ ]:
# Q3: Leaf node — Population
# Identify the actual leaf node(s) and their parents
leaf_name = leaf_nodes[0]  # Take the first leaf node
leaf_id = bn.idFromName(leaf_name)
leaf_parents = [bn.variable(p).name() for p in bn.parents(leaf_id)]

print("=" * 50)
print(f"LEAF NODE: {leaf_name}")
print("=" * 50)
print(f"Parents: {leaf_parents}")
print(f"Children: None (leaf)")
print(f"Number of parameters: {bn.cpt(leaf_name).domainSize()}")
print(f"\nCPT of {leaf_name} (leaf node):")
gnb.flow.row(bn.cpt(leaf_name), captions=[f"CPT of {leaf_name}"])


In [ ]:
# Q3: Intermediate and multi-parent node — HouseValue
# HouseValue has parents (Income, RoomSize, Occupancy) and child (HouseAge)
hv_id = bn.idFromName('HouseValue')
hv_parents = [bn.variable(p).name() for p in bn.parents(hv_id)]
hv_children = [bn.variable(c).name() for c in bn.children(hv_id)]

print("=" * 50)
print("INTERMEDIATE & MULTI-PARENT NODE: HouseValue")
print("=" * 50)
print(f"Parents: {hv_parents}")
print(f"Children: {hv_children}")
print(f"Number of parameters: {bn.cpt('HouseValue').domainSize()}")
print(f"\nCPT of HouseValue (intermediate node):")
gnb.flow.row(bn.cpt("HouseValue"), captions=["CPT of HouseValue"])


In [ ]:
# Q3: Additional intermediate node  Occupancy (if it has parents and children)
# This gives us a node with a single parent for complexity comparison
occ_id = bn.idFromName('Occupancy')
occ_parents = [bn.variable(p).name() for p in bn.parents(occ_id)]
occ_children = [bn.variable(c).name() for c in bn.children(occ_id)]

if occ_parents and occ_children:
    print("=" * 50)
    print("INTERMEDIATE NODE (single parent): Occupancy")
    print("=" * 50)
    print(f"Parents: {occ_parents}")
    print(f"Children: {occ_children}")
    print(f"Number of parameters: {bn.cpt('Occupancy').domainSize()}")
    print(f"\nCPT of Occupancy:")
    gnb.flow.row(bn.cpt("Occupancy"), captions=["CPT of Occupancy"])

# Summary of CPT complexity
print("\n" + "=" * 50)
print("CPT COMPLEXITY COMPARISON")
print("=" * 50)
for n in bn.nodes():
    name = bn.variable(n).name()
    n_parents = len(bn.parents(n))
    params = bn.cpt(n).domainSize()
    print(f"  {name}: {n_parents} parent(s) → {params} parameters")


## Question 4: Observational, Interventional, and Counterfactual Reasoning
Demonstrate three types of probabilistic reasoning using the learned BN.


### 4.1 Observational Reasoning

In [ ]:
# Q4.1: Observational — Prior distribution of HouseValue
ie = gum.LazyPropagation(bn)
ie.makeInference()
prior = ie.posterior("HouseValue")

print("Prior distribution P(HouseValue):")
print(prior)
gnb.flow.row(prior, captions=["Prior distribution P(HouseValue)"])


In [ ]:
# Q4.1: Observational — Posterior with Income=High observed
ie_obs = gum.LazyPropagation(bn)
ie_obs.setEvidence({"Income": "High"})
ie_obs.makeInference()
posterior_obs = ie_obs.posterior("HouseValue")

print("Posterior distribution P(HouseValue | Income=High):")
print(posterior_obs)

gnb.sideBySide(
    gnb.getInference(bn),
    gnb.getInference(bn, evs={"Income": "High"}),
    captions=["Prior (no evidence)", "Posterior (Income=High)"]
)


### 4.2 Interventional Reasoning

In [ ]:
# Q4.2: Interventional — Intervene on Occupancy (a NON-ROOT node)
# This is important: intervening on a non-root node produces DIFFERENT results
# from conditioning, because the do-operator severs incoming arcs.

# First: OBSERVATIONAL P(HouseValue | Occupancy=Low) — conditioning
ie_cond = gum.LazyPropagation(bn)
ie_cond.setEvidence({"Occupancy": "Low"})
ie_cond.makeInference()
obs_result = ie_cond.posterior("HouseValue")
print("OBSERVATIONAL: P(HouseValue | Occupancy=Low):")
print(obs_result)


In [ ]:
# Q4.2: Now compute INTERVENTIONAL P(HouseValue | do(Occupancy=Low))
# Build a causal model from the BN
causal_model = csl.CausalModel(bn)

# Display the causal diagram
gnb.show(causal_model)


In [ ]:
# Q4.2: Compute P(HouseValue | do(Occupancy=Low)) using pyAgrum's causal module
formula, impact, explanation = csl.causalImpact(
    causal_model,
    on="HouseValue",
    doing="Occupancy",
    values={"Occupancy": "Low"}
)

print("INTERVENTIONAL: P(HouseValue | do(Occupancy=Low)):")
print(impact)

# Show the causal impact with formula and explanation
cslnb.showCausalImpact(causal_model, on="HouseValue", doing="Occupancy",
                        values={"Occupancy": "Low"})


In [ ]:
# Q4.2: Side-by-side comparison — Observational vs Interventional
print("=" * 60)
print("COMPARISON: Observational vs Interventional on Occupancy")
print("=" * 60)
print(f"\nObservational P(HouseValue | Occupancy=Low):")
print(f"  {obs_result}")
print(f"\nInterventional P(HouseValue | do(Occupancy=Low)):")
print(f"  {impact}")

# Also do intervention on Income (root node) to show the degenerate case
formula_inc, impact_inc, explanation_inc = csl.causalImpact(
    causal_model,
    on="HouseValue",
    doing="Income",
    values={"Income": "High"}
)
print(f"\n--- For comparison: Root node intervention ---")
print(f"Observational P(HouseValue | Income=High):")
ie_inc = gum.LazyPropagation(bn)
ie_inc.setEvidence({"Income": "High"})
ie_inc.makeInference()
print(f"  {ie_inc.posterior('HouseValue')}")
print(f"Interventional P(HouseValue | do(Income=High)):")
print(f"  {impact_inc}")
print(f"\nNote: For root node Income, observational = interventional (no incoming arcs to sever)")


### 4.3 Counterfactual Reasoning

In [ ]:
# Q4.3: Counterfactual — Following the firing squad approach from course slides
# Factual world: We OBSERVE Occupancy=High, what is P(HouseValue)?

print("=" * 60)
print("COUNTERFACTUAL REASONING")
print("=" * 60)

# Factual observation
ie_factual = gum.LazyPropagation(bn)
ie_factual.setEvidence({"Occupancy": "High"})
ie_factual.makeInference()
factual_hv = ie_factual.posterior("HouseValue")
print(f"\nFACTUAL WORLD: P(HouseValue | Occupancy=High):")
print(factual_hv)


In [ ]:
# Q4.3: Counterfactual — What WOULD HAVE happened if Occupancy had been Low instead?
# Following the firing squad method: create a modified BN by emancipating Occupancy
# (removing its incoming arcs) and forcing Occupancy=Low

# Create a copy of the BN for the counterfactual world
bn_cf = gum.BayesNet(bn)

# Emancipate Occupancy: remove all incoming arcs to Occupancy
occ_id_cf = bn_cf.idFromName("Occupancy")
parents_of_occ = list(bn_cf.parents(occ_id_cf))
for parent_id in parents_of_occ:
    parent_name = bn_cf.variable(parent_id).name()
    bn_cf.eraseArc(parent_name, "Occupancy")
    print(f"Erased arc: {parent_name} -> Occupancy")

# Show structural difference between original and counterfactual BN
print(f"\nOriginal BN arcs: {bn.sizeArcs()}")
print(f"Counterfactual BN arcs: {bn_cf.sizeArcs()}")
gnb.showBNDiff(bn, bn_cf, size="6")


In [ ]:
# Q4.3: Counterfactual inference — Force Occupancy=Low in the modified BN
ie_cf = gum.LazyPropagation(bn_cf)
ie_cf.setEvidence({"Occupancy": "Low"})
ie_cf.makeInference()
cf_hv = ie_cf.posterior("HouseValue")
print(f"COUNTERFACTUAL WORLD: P(HouseValue | do(Occupancy=Low)) in modified BN:")
print(cf_hv)


In [ ]:
# Q4.3: Compare factual vs counterfactual worlds side by side
gnb.sideBySide(
    gnb.getInference(bn, evs={"Occupancy": "High"}),
    gnb.getInference(bn_cf, evs={"Occupancy": "Low"}),
    captions=[
        "Factual: P(· | Occupancy=High)",
        "Counterfactual: P(· | do(Occupancy=Low))"
    ]
)

print("\n" + "=" * 60)
print("COUNTERFACTUAL SUMMARY")
print("=" * 60)
print(f"\nFactual P(HouseValue | Occupancy=High):")
for i in range(factual_hv.variable(0).domainSize()):
    label = factual_hv.variable(0).label(i)
    print(f"  {label}: {factual_hv[i]:.4f}")

print(f"\nCounterfactual P(HouseValue | do(Occupancy=Low)):")
for i in range(cf_hv.variable(0).domainSize()):
    label = cf_hv.variable(0).label(i)
    print(f"  {label}: {cf_hv[i]:.4f}")


## Question 5: From Bayesian Networks to Influence Diagrams
Extend the BN into an Influence Diagram with 2 decision nodes and 2 utility nodes.


In [ ]:
# Q5: Build the Influence Diagram from scratch with the same chance nodes
# We rebuild the structure to include decision and utility nodes

diag = gum.InfluenceDiagram()

# --- Add chance nodes (same 6 variables as BN) ---
Income = diag.addChanceNode(gum.LabelizedVariable("Income", "Income", ["High", "Low", "Medium"]))
HouseAge = diag.addChanceNode(gum.LabelizedVariable("HouseAge", "HouseAge", ["Mid", "New", "Old"]))
RoomSize = diag.addChanceNode(gum.LabelizedVariable("RoomSize", "RoomSize", ["Large", "Medium", "Small"]))
Occupancy = diag.addChanceNode(gum.LabelizedVariable("Occupancy", "Occupancy", ["High", "Low", "Medium"]))
Population = diag.addChanceNode(gum.LabelizedVariable("Population", "Population", ["Dense", "Moderate", "Sparse"]))
HouseValue = diag.addChanceNode(gum.LabelizedVariable("HouseValue", "HouseValue", ["Cheap", "Expensive", "Moderate"]))

# --- Add decision nodes ---
InvestDecision = diag.addDecisionNode(gum.LabelizedVariable("InvestDecision", "InvestDecision", ["Buy", "Hold", "Sell"]))
RenovateDecision = diag.addDecisionNode(gum.LabelizedVariable("RenovateDecision", "RenovateDecision", ["Renovate", "NoRenovate"]))

# --- Add utility nodes ---
FinancialReturn = diag.addUtilityNode(gum.LabelizedVariable("FinancialReturn", "FinancialReturn", 1))
RiskCost = diag.addUtilityNode(gum.LabelizedVariable("RiskCost", "RiskCost", 1))

# --- Add arcs from the learned BN structure ---
# Reproduce all 12 arcs from the learned BN
for arc in bn.arcs():
    tail_name = bn.variable(arc[0]).name()
    head_name = bn.variable(arc[1]).name()
    diag.addArc(tail_name, head_name)

# --- Add information arcs to decision nodes ---
# InvestDecision observes Income and HouseValue before deciding
diag.addArc("Income", "InvestDecision")
diag.addArc("HouseValue", "InvestDecision")

# RenovateDecision observes HouseAge and InvestDecision (temporal ordering)
diag.addArc("HouseAge", "RenovateDecision")
diag.addArc("InvestDecision", "RenovateDecision")

# --- Add arcs to utility nodes ---
# FinancialReturn depends on InvestDecision and HouseValue
diag.addArc("InvestDecision", "FinancialReturn")
diag.addArc("HouseValue", "FinancialReturn")

# RiskCost depends on RenovateDecision, HouseAge, and Occupancy
diag.addArc("RenovateDecision", "RiskCost")
diag.addArc("HouseAge", "RiskCost")
diag.addArc("Occupancy", "RiskCost")

print("Influence Diagram built successfully!")
print(f"Chance nodes: {[diag.variable(n).name() for n in diag.nodes() if diag.isChanceNode(n)]}")
print(f"Decision nodes: {[diag.variable(n).name() for n in diag.nodes() if diag.isDecisionNode(n)]}")
print(f"Utility nodes: {[diag.variable(n).name() for n in diag.nodes() if diag.isUtilityNode(n)]}")


In [ ]:
# Q5: Transfer learned CPTs from BN to ID chance nodes
# This is CRITICAL — without this, the ID uses uninformative uniform priors

for n in diag.nodes():
    if diag.isChanceNode(n):
        name = diag.variable(n).name()
        if name in bn.names():
            # Get BN CPT and copy to ID
            bn_cpt = bn.cpt(name)
            id_cpt = diag.cpt(name)

            # Check if dimensions match
            if bn_cpt.domainSize() == id_cpt.domainSize():
                id_cpt.fillWith(bn_cpt)
                print(f"  ✓ Transferred CPT for {name} ({bn_cpt.domainSize()} params)")
            else:
                print(f"  ✗ Dimension mismatch for {name}: BN={bn_cpt.domainSize()}, ID={id_cpt.domainSize()}")
                # Manual copy with matching variable order
                diag.cpt(name).fillWith(bn.cpt(name), bn.names())
                print(f"    → Transferred using name matching")

# Verify transfer by comparing a CPT
print(f"\nVerification — Income CPT in BN: {bn.cpt('Income')[:].tolist()}")
print(f"Verification — Income CPT in ID: {diag.cpt('Income')[:].tolist()}")
print(f"\nCPT transfer to ID: {'VERIFIED' if np.allclose(bn.cpt('Income')[:], diag.cpt('Income')[:]) else 'FAILED'}")


In [ ]:
# Q5: Define utility tables

# --- FinancialReturn utility: depends on InvestDecision × HouseValue ---
# Rows: InvestDecision (Buy/Hold/Sell), Cols: HouseValue (Cheap/Expensive/Moderate)
#
# Logic:
#   Buy + Cheap = high return (buy low, sell high potential) → 80
#   Buy + Moderate = decent return → 40
#   Buy + Expensive = overpaying risk → -10
#   Hold + Cheap = stable but missed opportunity → 20
#   Hold + Moderate = safe and steady → 40
#   Hold + Expensive = good asset retention → 50
#   Sell + Cheap = poor return on sale → 10
#   Sell + Moderate = reasonable exit → 30
#   Sell + Expensive = excellent exit → 90

diag.utility("FinancialReturn")[{"InvestDecision": "Buy", "HouseValue": "Cheap"}] = [80]
diag.utility("FinancialReturn")[{"InvestDecision": "Buy", "HouseValue": "Moderate"}] = [40]
diag.utility("FinancialReturn")[{"InvestDecision": "Buy", "HouseValue": "Expensive"}] = [-10]
diag.utility("FinancialReturn")[{"InvestDecision": "Hold", "HouseValue": "Cheap"}] = [20]
diag.utility("FinancialReturn")[{"InvestDecision": "Hold", "HouseValue": "Moderate"}] = [40]
diag.utility("FinancialReturn")[{"InvestDecision": "Hold", "HouseValue": "Expensive"}] = [50]
diag.utility("FinancialReturn")[{"InvestDecision": "Sell", "HouseValue": "Cheap"}] = [10]
diag.utility("FinancialReturn")[{"InvestDecision": "Sell", "HouseValue": "Moderate"}] = [30]
diag.utility("FinancialReturn")[{"InvestDecision": "Sell", "HouseValue": "Expensive"}] = [90]

# --- RiskCost utility: depends on RenovateDecision × HouseAge × Occupancy ---
# Renovating old, crowded buildings is expensive and risky
# Not renovating avoids cost but old buildings degrade

for ha in ["Mid", "New", "Old"]:
    for occ in ["High", "Low", "Medium"]:
        # Renovation costs
        if ha == "Old" and occ == "High":
            ren_cost = -50  # worst case: old + crowded
        elif ha == "Old" and occ == "Medium":
            ren_cost = -35
        elif ha == "Old" and occ == "Low":
            ren_cost = -20
        elif ha == "Mid" and occ == "High":
            ren_cost = -30
        elif ha == "Mid" and occ == "Medium":
            ren_cost = -15
        elif ha == "Mid" and occ == "Low":
            ren_cost = -5
        elif ha == "New" and occ == "High":
            ren_cost = -15
        elif ha == "New" and occ == "Medium":
            ren_cost = -5
        else:  # New + Low
            ren_cost = 0

        # No renovation: degradation penalty for old buildings
        if ha == "Old":
            no_ren_cost = -10  # degradation penalty
        elif ha == "Mid":
            no_ren_cost = -3
        else:
            no_ren_cost = 0

        diag.utility("RiskCost")[{"RenovateDecision": "Renovate", "HouseAge": ha, "Occupancy": occ}] = [ren_cost]
        diag.utility("RiskCost")[{"RenovateDecision": "NoRenovate", "HouseAge": ha, "Occupancy": occ}] = [no_ren_cost]

# Display utility tables
print("FinancialReturn utility table:")
gnb.flow.row(diag.utility("FinancialReturn"), captions=["U[FinancialReturn | InvestDecision, HouseValue]"])
print("\nRiskCost utility table:")
gnb.flow.row(diag.utility("RiskCost"), captions=["U[RiskCost | RenovateDecision, HouseAge, Occupancy]"])


In [ ]:
# Q5: Display the complete Influence Diagram
gnb.showInfluenceDiagram(diag, size="10")


## Question 6: Decision Analysis
Test decision scenarios, compute optimal policy and MEU.


In [ ]:
# Q6: Scenario 1 — Buy + Renovate
print("=" * 60)
print("SCENARIO 1: Buy + Renovate")
print("=" * 60)
result_s1 = gnb.getInference(diag, evs={"InvestDecision": "Buy", "RenovateDecision": "Renovate"})
gnb.showInference(diag, evs={"InvestDecision": "Buy", "RenovateDecision": "Renovate"}, size="10")


In [ ]:
# Q6: Scenario 2 — Hold + NoRenovate
print("=" * 60)
print("SCENARIO 2: Hold + NoRenovate")
print("=" * 60)
gnb.showInference(diag, evs={"InvestDecision": "Hold", "RenovateDecision": "NoRenovate"}, size="10")


In [ ]:
# Q6: Compute optimal decisions using LIMID solver
print("=" * 60)
print("OPTIMAL DECISIONS (LIMID Solver)")
print("=" * 60)

ie_id = gum.ShaferShenoyLIMIDInference(diag)
ie_id.makeInference()

# Show the full inference with optimal policy
gnb.showInference(diag, engine=ie_id, size="12")


In [ ]:
# Q6: Exhaustive comparison of ALL 6 decision combinations
print("=" * 60)
print("EXHAUSTIVE COMPARISON: All 6 Decision Combinations")
print("=" * 60)

invest_options = ["Buy", "Hold", "Sell"]
renovate_options = ["Renovate", "NoRenovate"]

results = []
for invest in invest_options:
    for renovate in renovate_options:
        ie_test = gum.ShaferShenoyLIMIDInference(diag)
        ie_test.setEvidence({"InvestDecision": invest, "RenovateDecision": renovate})
        ie_test.makeInference()
        meu = ie_test.meanVar("FinancialReturn")
        risk = ie_test.meanVar("RiskCost")
        total_mean = meu["mean"] + risk["mean"]
        total_var = meu["variance"] + risk["variance"]
        total_stdev = np.sqrt(max(0, total_var))
        results.append({
            "InvestDecision": invest,
            "RenovateDecision": renovate,
            "EU": round(total_mean, 2),
            "stdev": round(total_stdev, 2)
        })
        print(f"  {invest:4s} + {renovate:11s}  :  EU = {total_mean:7.2f}  (stdev = {total_stdev:.2f})")

# Find best and worst
best = max(results, key=lambda x: x["EU"])
worst = min(results, key=lambda x: x["EU"])
print(f"\nBest:  {best['InvestDecision']} + {best['RenovateDecision']} (EU = {best['EU']})")
print(f"Worst: {worst['InvestDecision']} + {worst['RenovateDecision']} (EU = {worst['EU']})")


In [ ]:
# Q6: Show all 6 scenarios side by side (visual comparison)
gnb.sideBySide(
    gnb.getInference(diag, evs={"InvestDecision": "Buy", "RenovateDecision": "Renovate"}),
    gnb.getInference(diag, evs={"InvestDecision": "Buy", "RenovateDecision": "NoRenovate"}),
    gnb.getInference(diag, evs={"InvestDecision": "Hold", "RenovateDecision": "Renovate"}),
    gnb.getInference(diag, evs={"InvestDecision": "Hold", "RenovateDecision": "NoRenovate"}),
    gnb.getInference(diag, evs={"InvestDecision": "Sell", "RenovateDecision": "Renovate"}),
    gnb.getInference(diag, evs={"InvestDecision": "Sell", "RenovateDecision": "NoRenovate"}),
    ncols=2,
    captions=[
        "Buy + Renovate", "Buy + NoRenovate",
        "Hold + Renovate", "Hold + NoRenovate",
        "Sell + Renovate", "Sell + NoRenovate"
    ]
)


## Question 7: Critical Evaluation
Model evaluation summary and statistics.


In [ ]:
# Q7: Model evaluation summary
print("=" * 60)
print("MODEL EVALUATION SUMMARY")
print("=" * 60)

print(f"\nBayesian Network:")
print(f"  Nodes: {bn.size()}")
print(f"  Arcs: {bn.sizeArcs()}")
print(f"  Max indegree: {max(len(bn.parents(n)) for n in bn.nodes())}")
print(f"  Total CPT parameters: {sum(bn.cpt(n).domainSize() for n in bn.nodes())}")
print(f"  Data rows: 20640")

print(f"\nInfluence Diagram:")
print(f"  Chance: {sum(1 for n in diag.nodes() if diag.isChanceNode(n))}")
print(f"  Decision: {sum(1 for n in diag.nodes() if diag.isDecisionNode(n))}")
print(f"  Utility: {sum(1 for n in diag.nodes() if diag.isUtilityNode(n))}")
print(f"  Total arcs: {diag.sizeArcs()}")

# Key results
ie_prior = gum.LazyPropagation(bn)
ie_prior.makeInference()
prior_hv = ie_prior.posterior("HouseValue")

ie_post = gum.LazyPropagation(bn)
ie_post.setEvidence({"Income": "High"})
ie_post.makeInference()
post_hv = ie_post.posterior("HouseValue")

print(f"\nKey Results:")
print(f"  Prior P(HouseValue=Expensive): {prior_hv.tolist()}")
print(f"  Posterior P(HouseValue=Expensive | Income=High): {post_hv.tolist()}")

ie_opt = gum.ShaferShenoyLIMIDInference(diag)
ie_opt.makeInference()
print(f"  MEU (optimal): {ie_opt.meanVar('FinancialReturn')['mean'] + ie_opt.meanVar('RiskCost')['mean']:.2f}")

# Verify CPT transfer
match = np.allclose(bn.cpt("Income")[:], diag.cpt("Income")[:])
print(f"  CPT transfer to ID: {'VERIFIED' if match else 'FAILED'}")
